# 03 Adaptive conformal inference (ACI)

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from src.backtests import backtest_summary
from src.pipeline import build_models

m = build_models()
loss = m['loss']
sigma_ewma = m['sigma_ewma']

## ACI with an EWMA base model

In [ ]:
from src.risk_measures import hs_var_es, ewma_var_es, fhs_var_es, aci_ewma_var

# Main sample: 250-day window, lambda = 0.94
hs_main   = hs_var_es(loss, window=250)
ewma_main = ewma_var_es(sigma_ewma)
fhs_main  = fhs_var_es(loss, sigma_ewma, window=250)
aci_main  = aci_ewma_var(loss, sigma_ewma)

common_main = (hs_main['var'].dropna().index
               .intersection(ewma_main['VaR'].dropna().index)
               .intersection(fhs_main['VaR'].dropna().index))
print(len(common_main))  

In [ ]:
from src.risk_measures import aci_ewma_var


aci = aci_ewma_var(loss, sigma_ewma)
aci_c = aci.loc[common_main]
print(aci_c.dropna().head())
print(aci_c['a'].describe())

In [ ]:
hits_aci = (loss[common_main] > aci_c['VaR']).astype(int)
print(hits_aci.sum())   

In [ ]:
backtest_summary(hits_aci, "ACI")

In [ ]:
results = {}
hits_robustness_aci = {}
gammas = [0.001, 0.005, 0.01, 0.02, 0.05]
summaries_robustness_aci = {}
for g in gammas:
    results[g] = aci_ewma_var(loss, sigma_ewma, gamma=g)

    results[g] = results[g].loc[common_main]

    hits_robustness_aci[g] = (loss.loc[common_main] > results[g]['VaR']).astype(int)

    summaries_robustness_aci[g] = backtest_summary(hits_robustness_aci[g], f'ACI_{g}')

    n_inf = np.isinf(results[g]['VaR']).sum()                                  
    avg_var = results[g]['VaR'].replace(np.inf, np.nan).mean()         

    summaries_robustness_aci[g]['n_inf'] = n_inf
    summaries_robustness_aci[g]['avg_VaR'] = avg_var

summaries_robustness_aci = pd.concat(summaries_robustness_aci.values())

display((summaries_robustness_aci))

print(fhs_main['VaR'].loc[common_main].mean())

## ACI with an FHS base model

In [ ]:
from src.risk_measures import aci_fhs_var

results = {}
hits_robustness_aci = {}
gammas = [0.001, 0.005, 0.01, 0.02, 0.05]
summaries_robustness_aci = {}
for g in gammas:
    results[g] = aci_fhs_var(loss, sigma_ewma, gamma=g)

    results[g] = results[g].loc[common_main]

    hits_robustness_aci[g] = (loss.loc[common_main] > results[g]['VaR']).astype(int)

    summaries_robustness_aci[g] = backtest_summary(hits_robustness_aci[g], f'ACI_FHS_{g}')

    n_inf = np.isinf(results[g]['VaR']).sum()                                  
    avg_var = results[g]['VaR'].replace(np.inf, np.nan).mean()         

    summaries_robustness_aci[g]['n_inf'] = n_inf
    summaries_robustness_aci[g]['avg_VaR'] = avg_var

summaries_robustness_aci = pd.concat(summaries_robustness_aci.values())

display((summaries_robustness_aci))

print(fhs_main['VaR'].loc[common_main].mean())

In [ ]:
a_gamma_0001 = aci_fhs_var(loss, sigma_ewma, gamma=0.001)['a']

plt.figure()
plt.plot(a_gamma_0001, label = r'$a_t$', color = 'grey')
plt.axhline(0.01, color='orange', linestyle='-.', label='Target miss rate (1%)')
plt.title(r'ACI-FHS: adaptive miss level $a_t$ ($\gamma = 0.001$)')
plt.xlabel('Time')
plt.ylabel(r'$a_t$ (adaptive miss level)')
plt.legend()
plt.savefig(f'{ROOT}\\results\\aci_fhs_alpha_over_time.png')
plt.show()

## Out-of-sample choice of γ

In [ ]:
mid = common_main[len(common_main) // 2]
print(mid)
train = common_main[common_main < mid]
test = common_main[common_main >= mid]

gammas = [0.001, 0.005, 0.01, 0.02, 0.05]
rows = {}

for g in gammas:
    aci = aci_fhs_var(loss, sigma_ewma, gamma = g)  
    var_c = aci['VaR'].loc[train]
    hits_c = (loss.loc[train] > var_c)
    rows[g] = {'breach_rate': hits_c.mean(), 'n_inf': np.isinf(var_c).sum()}

train_table = pd.DataFrame(rows).T
train_table

gamma_chosen = float((train_table[train_table['n_inf'] == 0]
                      ['breach_rate'] - 0.01).abs().idxmin())

c_test = test
models = {'FHS': fhs_main['VaR'],
          'ACI γ=0.005': aci_fhs_var(loss, sigma_ewma, gamma=gamma_chosen)['VaR'],
          'ACI γ=0.001': aci_fhs_var(loss, sigma_ewma, gamma=0.001)['VaR']}

res = {}
for name, var in models.items():
    v = var.loc[c_test]
    hits = (loss.loc[c_test] > v).astype(int)
    s = backtest_summary(hits, name)
    s['n_inf'] = np.isinf(v).sum()
    res[name] = s

pd.concat(res.values())

v = models['ACI γ=0.005'].loc[c_test]
print(v[np.isinf(v)].index.year.value_counts().sort_index())